In [1]:
# System dependency for MIDI synthesis
!apt-get install -y fluidsynth

# Download a SoundFont for MIDI -> audio conversion
!wget https://github.com/urish/c-flute/raw/master/font/GeneralUser%20GS%20v1.471.sf2 -O /content/font.sf2

# Python packages
!pip install torch torchaudio transformers librosa numpy pandas matplotlib seaborn scikit-learn pretty_midi pyfluidsynth ipython tqdm

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core fluid-soundfont-gm gsettings-desktop-schemas libatk-bridge2.0-0
  libatk1.0-0 libatk1.0-data libatspi2.0-0 libdouble-conversion3 libevdev2
  libfluidsynth3 libgtk-3-0 libgtk-3-bin libgtk-3-common libgudev-1.0-0
  libinput-bin libinput10 libinstpatch-1.0-2 libmd4c0 libmtdev1 libqt5core5a
  libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5
  librsvg2-common libwacom-bin libwacom-common libwacom9 libxcb-icccm4
  libxcb-image0 libxcb-keysyms1 libxcb-render-util0 libxcb-util1
  libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1 libxcomposite1
  libxkbcommon-x11-0 libxtst6 qsynth qt5-gtk-platformtheme
  qttranslations5-l10n session-migration timgm6mb-soundfont
Suggested packages:
  fluid-soundfont-gs gvfs qt5-image-formats-plugins qtwayland5 jackd
The following NEW packages will be installed:
  at-spi2-core fluid-soundfont

In [2]:

import os
import random
import torch
import numpy as np
import pandas as pd
import librosa
import pretty_midi
from transformers import AutoModel, AutoProcessor
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from IPython.display import Audio, display
import warnings
import os
from IPython.display import Audio, display

In [3]:

drive.mount('/content/drive')

Mounted at /content/drive


## Path Setup Guide

In your Google Drive create a folder in MyDrive called "AAI-590 Capstone"

Go to Israel's shared drive:  https://drive.google.com/drive/u/1/folders/1nIY6vsEoQIsdz2RacIIng0QBS-NvrF9U

Click the ...

Select "Organize"->Add Shortcut->All Locations-> /MyDrive/AAI-590 Capstone/

In [12]:
# Path to ChunkSamples folder
# (which now has a shortcut to your MyDrive per instructions above)
chunk_path = "/content/drive/MyDrive/AAI-590 Capstone/ChunkSamples"

In [ ]:

# List subdirectories
os.listdir(chunk_path)

# Create triplet dataframe

##### Loops through chunk_path directory and extract file names for dataframe of filenames only

Anchor: first chunk of every song and every composer.

Positive: all other chunks from the same song.

Negative: chunks from one randomly chosen different composer, number of

negatives matched to positives, ensuring no repeats globally.

In [25]:


data = []

# Function to sort filenames numerically by chunk number
def numeric_sort(f):
    match = re.search(r'_chunk_(\d+)\.wav$', f)
    return int(match.group(1)) if match else -1

# Keep track of negatives already used
used_negatives = set()

# Loop through each composer
for composer in os.listdir(chunk_path):
    composer_path = os.path.join(chunk_path, composer)
    if not os.path.isdir(composer_path):
        continue

    # Loop through each song of the current composer
    for song in os.listdir(composer_path):
        song_path = os.path.join(composer_path, song)
        if not os.path.isdir(song_path):
            continue

        # Get chunks for this song, sorted numerically
        chunks = sorted([f for f in os.listdir(song_path) if f.endswith(".wav")], key=numeric_sort)
        if len(chunks) < 2:
            continue

        anchor = chunks[0]
        positives = chunks[1:]  # sorted numerically

        # Pick one OTHER composer at random
        other_composers = [c for c in os.listdir(chunk_path) if c != composer and os.path.isdir(os.path.join(chunk_path, c))]
        if not other_composers:
            continue
        chosen_composer = random.choice(other_composers)
        chosen_composer_path = os.path.join(chunk_path, chosen_composer)

        # Collect all chunks from all songs of that composer, sorted numerically
        negative_pool = []
        for other_song in sorted(os.listdir(chosen_composer_path)):
            other_song_path = os.path.join(chosen_composer_path, other_song)
            if not os.path.isdir(other_song_path):
                continue
            for f in sorted(os.listdir(other_song_path), key=numeric_sort):
                if f.endswith(".wav") and f not in used_negatives:
                    negative_pool.append(f)

        # Match lengths: positives vs negatives
        min_len = min(len(positives), len(negative_pool))
        for i in range(min_len):
            data.append({
                "anchor": anchor,
                "positive": positives[i],
                "negative": negative_pool[i]
            })
            used_negatives.add(negative_pool[i])

# Create dataframe
df = pd.DataFrame(data)

# Show head
print("Dataframe head:")
print(df.head())

Dataframe head:
                anchor             positive                 negative
0  alb_se7_chunk_1.wav  alb_se7_chunk_2.wav  schum_abegg_chunk_1.wav
1  alb_se7_chunk_1.wav  alb_se7_chunk_3.wav  schum_abegg_chunk_2.wav
2  alb_se7_chunk_1.wav  alb_se7_chunk_4.wav  schum_abegg_chunk_3.wav
3  alb_se7_chunk_1.wav  alb_se7_chunk_5.wav  schum_abegg_chunk_4.wav
4  alb_se7_chunk_1.wav  alb_se7_chunk_6.wav  schum_abegg_chunk_5.wav


In [23]:
# Check uniqueness and counts
# Check uniqueness and counts
unique_positives = df['positive'].unique()
unique_negatives = df['negative'].unique()
print("\nPositive column unique names:", len(unique_positives) == len(df['positive']))
print("Total positive entries:", len(df['positive']))
print("Total unique positive filenames:", len(unique_positives))
# Total dataframe counts
print("-" * 50)
# Total files in chunk_path
total_files = sum(len(files) for _, _, files in os.walk(chunk_path))
print("Total files in chunk_path:", total_files)
print("Total number of rows in dataframe:", len(df))
print("Total values in anchor column:", df['anchor'].count())
print("Total values in positive column:", df['positive'].count())
print("Total values in negative column:", df['negative'].count())
print("-" * 50)
print("Negative column unique names:", len(unique_negatives) == len(df['negative']))
print("Total negative entries:", len(df['negative']))
print("Total unique negative filenames:", len(unique_negatives))
print("-"*50)
print("Anchor filename counts (per unique anchor):")
print(df['anchor'].value_counts())
print("-"*50)
print("Total number of rows in dataframe:", len(df))

# Count for each unique anchor filename
anchor_counts = df['anchor'].value_counts()
print("Anchor filename counts (per unique anchor):")
print(anchor_counts)






Positive column unique names: True
Total positive entries: 21829
Total unique positive filenames: 21829
--------------------------------------------------
Total files in chunk_path: 43663
Total number of rows in dataframe: 21829
Total values in anchor column: 21829
Total values in positive column: 21829
Total values in negative column: 21829
--------------------------------------------------
Negative column unique names: True
Total negative entries: 21829
Total unique negative filenames: 21829
--------------------------------------------------
Anchor filename counts (per unique anchor):
anchor
liz_donjuan_chunk_1.wav                  1074
beethoven_hammerklavier_1_chunk_1.wav     516
islamei_chunk_1.wav                       500
beethoven_hammerklavier_4_chunk_1.wav     498
appass_3_chunk_1.wav                      449
                                         ... 
chpn-p1_chunk_1.wav                        18
chpn-p11_chunk_1.wav                       18
haydn_8_2_chunk_1.wav         

# Add MERT baseline positive and negative scores to the dataframe

In [26]:
########################################
# Section 0 - Load MERT model and libraries
########################################



model_name = "m-a-p/MERT-v1-95M"

processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
model.eval()
print("MERT model loaded successfully")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/211 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_MERT.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-95M:
- configuration_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
The image processor of type `Wav2Vec2ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.


modeling_MERT.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-95M:
- modeling_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

MERT model loaded successfully


In [ ]:



########################################
# Section 1A - Load .wav → embeddings
########################################

def wav_to_audio(wav_path, sr=24000):
    """Load .wav file as waveform"""
    audio, sr = librosa.load(wav_path, sr=sr)
    return audio, sr

def embed_audio(audio, sr=24000):
    """Compute MERT embedding for a waveform"""
    inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    emb = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return emb

def cosine_sim(a, b):
    """Compute cosine similarity between embeddings"""
    return float(cosine_similarity(a.reshape(1,-1), b.reshape(1,-1))[0][0])

########################################
# Section 1B - Compute positive/negative MERT similarity scores
# with full paths and truncation to 4 decimals
########################################

positive_scores = []
negative_scores = []

# Build a lookup dict for full path
file_lookup = {}
for composer in os.listdir(chunk_path):
    composer_path = os.path.join(chunk_path, composer)
    if not os.path.isdir(composer_path):
        continue
    for song in os.listdir(composer_path):
        song_path = os.path.join(composer_path, song)
        if not os.path.isdir(song_path):
            continue
        for f in os.listdir(song_path):
            if f.endswith(".wav"):
                file_lookup[f] = os.path.join(song_path, f)  # filename → full path

# Iterate over DataFrame rows
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Computing MERT scores"):
    anchor_path = file_lookup[row["anchor"]]
    pos_path    = file_lookup[row["positive"]]
    neg_path    = file_lookup[row["negative"]]

    # Load audio and compute embeddings
    anchor_audio, sr = wav_to_audio(anchor_path)
    pos_audio, sr    = wav_to_audio(pos_path)
    neg_audio, sr    = wav_to_audio(neg_path)

    anchor_emb = embed_audio(anchor_audio, sr)
    pos_emb    = embed_audio(pos_audio, sr)
    neg_emb    = embed_audio(neg_audio, sr)

    # Compute cosine similarities and truncate
    positive_scores.append(round(cosine_sim(anchor_emb, pos_emb), 4))
    negative_scores.append(round(cosine_sim(anchor_emb, neg_emb), 4))

# Add scores to DataFrame
df["positive_base_score"] = positive_scores
df["negative_base_score"] = negative_scores

# Show head
print(df.head())

Computing MERT scores:   0%|          | 75/24462 [10:14<55:20:30,  8.17s/it]

In [ ]:
# Export DataFrame with MERT scores to Drive (inside ChunkSamples folder)
export_path = os.path.join(chunk_path, "chunk_dataset_with_scores.csv")
df.to_csv(export_path, index=False)
print(f"DataFrame exported successfully to: {export_path}")

In [ ]:
# Show max and min for the new MERT score columns
print(f"Positive base score → max: {df['positive_base_score'].max():.4f}, min: {df['positive_base_score'].min():.4f}")
print(f"Negative base score → max: {df['negative_base_score'].max():.4f}, min: {df['negative_base_score'].min():.4f}")

In [ ]:


########################################
# Positive threshold search
########################################
threshold = float(input("Enter desired positive score for evaluation (0 to 1): "))

# Find row with closest positive_base_score
closest_idx = (df['positive_base_score'] - threshold).abs().idxmin()
closest_row = df.loc[closest_idx]

print(f"\nThe row closest to input score {threshold} is row index: {closest_idx}")
print(f"Anchor file: {closest_row['anchor']}")
print(f"Positive file: {closest_row['positive']}")
print(f"Positive base score: {closest_row['positive_base_score']}")

# Reconstruct full paths (assumes df has composer/song info or you can infer from folder structure)
anchor_path = os.path.join(chunk_path, closest_row['anchor_composer'], closest_row['anchor_song'], closest_row['anchor'])
positive_path = os.path.join(chunk_path, closest_row['anchor_composer'], closest_row['anchor_song'], closest_row['positive'])

print("\n--- Playing Anchor ---")
display(Audio(anchor_path, autoplay=False))
print("\n--- Playing Positive ---")
display(Audio(positive_path, autoplay=False))



In [ ]:
########################################
# Negative threshold search
########################################
neg_threshold = float(input("Enter desired negative score for evaluation (0 to 1): "))

# Find row with closest negative_base_score
closest_neg_idx = (df['negative_base_score'] - neg_threshold).abs().idxmin()
closest_neg_row = df.loc[closest_neg_idx]

print(f"\nThe row closest to negative input score {neg_threshold} is row index: {closest_neg_idx}")
print(f"Anchor file: {closest_neg_row['anchor']}")
print(f"Negative file: {closest_neg_row['negative']}")
print(f"Negative base score: {closest_neg_row['negative_base_score']}")

# Reconstruct full paths
anchor_neg_path = os.path.join(chunk_path, closest_neg_row['anchor_composer'], closest_neg_row['anchor_song'], closest_neg_row['anchor'])
negative_path = os.path.join(chunk_path, closest_neg_row['negative_composer'], closest_neg_row['negative_song'], closest_neg_row['negative'])

print("\n--- Playing Anchor ---")
display(Audio(anchor_neg_path, autoplay=False))
print("\n--- Playing Negative ---")
display(Audio(negative_path, autoplay=False))